<a href="https://colab.research.google.com/github/ard0p8v/KSU2_SatelliteImageClassification/blob/main/SatelliteImageClassification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Zápočtový úkol 2 - Klasifikace využití půdy ze satelitních snímků EuroSAT pomocí transfer learningu
**Předmět:** Strojové učení II (KSU2)  
**Autor:** Bc. Pavel Ardolf  
**GitHub Repozitář:** https://github.com/ard0p8v/KSU2_SatelliteImageClassification

---

## 1. Cíl projektu
Cílem práce je návrh a implementace hluboké konvoluční neuronové sítě pro automatickou klasifikaci satelitních snímků z programu ESA Sentinel-2.[1] Projekt demonstruje schopnost aplikace metody **přeneseného učení (Transfer Learning)** a techniky **jemného doladění (Fine-tuning)** pro efektivní rozpoznávání vzorů v datech dálkového průzkumu Země.[2]

## 2. Popis dat
V úloze je využit dataset **EuroSAT**, který představuje standard pro klasifikaci využití půdy (Land Use / Land Cover):[1]
* **Obsah:** Celkem 27 000 digitálně označených snímků v RGB spektru.
* **Struktura:** Dataset je rozdělen do 10 vyvážených tříd (3 000 vzorků na třídu), což zajišťuje stabilitu trénovacího procesu.
* **Třídy:** Zahrnují kategorie jako jsou lesy (**Forest**), průmyslové zóny (**Industrial**), dálnice (**Highway**) či obytné oblasti (**Residential**).
* **Rozlišení:** Každý snímek má velikost 64x64 pixelů, což odpovídá multispektrálnímu rozlišení satelitů Sentinel-2.

## 3. Metodika
V notebooku jsou realizovány kroky v souladu s pokročilými postupy pro hluboké učení a látkou probíranou v přednáškách SU2:[1][2][3]
1. **Příprava dat a EDA:** Načtení dat přes `tensorflow_datasets`, vizualizace distribuce tříd a ukázkových snímků.
2. **Augmentace dat:** Implementace vrstev pro náhodnou rotaci, překlápění a úpravu jasu/kontrastu za účelem zvýšení robustnosti modelu a potlačení overfittingu.
3. **Modelování (Transfer Learning):**
    - **Fáze 1 (Feature Extraction):** Využití předtrénované architektury **MobileNetV2** [4] se zmrazenými vahami pro extrakci příznaků a natrénování nové klasifikační hlavy.
    - **Fáze 2 (Fine-tuning):** Rozmrazení hlubokých konvolučních vrstev a jejich jemné doladění s nízkým učícím koeficientem.
4. **Evaluace:** Komplexní posouzení kvality pomocí testovací přesnosti (Accuracy), matice záměn (**Confusion Matrix**) a metriky F1-score pro jednotlivé třídy.

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import tensorflow_datasets as tfds

# Konfigurace cest a parametrů ze zdroje
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

BATCH_SIZE = 32
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

CLASS_NAMES_CZ = [
    "Jednoletá plodina", "Les", "Bylinná vegetace",
    "Dálnice", "Průmysl", "Pastviny",
    "Trvalá plodina", "Obytná zóna", "Řeka", "Moře/jezero"
]

print(f"TensorFlow verze: {tf.__version__}")

TensorFlow verze: 2.19.0


## 4. Příprava dat a EDA
Před samotným trénováním provádíme **Exploratory Data Analysis (EDA)**. Dataset EuroSAT je balancovaný, což znamená, že každá z 10 tříd obsahuje přesně 3 000 vzorků.[1] Implementujeme tedy pipeline pro zpracování dat:[7]

* **Normalizace:** Snímky jsou transformovány pro model MobileNetV2 do intervalu [-1, 1].
* **Augmentace:** Pro zvýšení generalizační schopnosti a potlačení overfittingu aplikujeme náhodné rotace, překlopení a úměrnou změnu jasu.[1]
* **Rozdělení:** Data jsou rozdělena na trénovací (70 %), validační (15 %) a testovací (15 %) část.[7]

In [11]:
# Místo tfds.load použij Hugging Face datasets
!pip install datasets -q

from datasets import load_dataset
import numpy as np
import tensorflow as tf

hf_ds = load_dataset("blanchon/EuroSAT_RGB", split="train")

CLASS_NAMES = hf_ds.features["label"].names  # 10 tříd v správném pořadí

def hf_to_tf(example):
    img = np.array(example["image"])           # PIL → numpy, tvar (64,64,3)
    lbl = example["label"]
    return img, lbl

# Převod na tf.data.Dataset
imgs = np.stack([np.array(e["image"]) for e in hf_ds])
lbls = np.array([e["label"] for e in hf_ds])

full_ds = tf.data.Dataset.from_tensor_slices((imgs, lbls)).shuffle(27000, seed=42)

n = len(full_ds)
n_train = int(n * 0.70)
n_val   = int(n * 0.15)

ds_train_raw = full_ds.take(n_train)
ds_val_raw   = full_ds.skip(n_train).take(n_val)
ds_test_raw  = full_ds.skip(n_train + n_val)

print(f"Trénovací: {n_train} | Validační: {n_val} | Testovací: {n - n_train - n_val}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/105M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/34.8M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/34.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16200 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5400 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5400 [00:00<?, ? examples/s]

Trénovací: 11340 | Validační: 2430 | Testovací: 2430
